In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import matplotlib.pyplot as plt
from os import listdir
import time    
import os
from tqdm import tqdm
%matplotlib inline

In [10]:
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return f"{h}:{m}:{round(s,1)}"

In [11]:
def augment_data(file_dir, n_generated_samples, save_to_dir):
    data_gen = ImageDataGenerator(rotation_range=15, 
                                  width_shift_range=0.3, 
                                  height_shift_range=0.3, 
                                  horizontal_flip=True, 
                                  vertical_flip=True, 
                                  fill_mode='nearest'
                                 )

    
    for filename in listdir(file_dir):
        # load the image
    
        image = cv2.imread(file_dir + '/' + filename)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # reshape the image
        image = image.reshape((1,)+image.shape)
        # prefix of the names for the generated sampels.
        save_prefix = 'aug_' + filename[:-4]
        # generate 'n_generated_samples' sample images
        i=0
        for batch in data_gen.flow(x=image, batch_size=1, save_to_dir=save_to_dir, 
                                           save_prefix=save_prefix, save_format='png'):
            i += 1
            if i > n_generated_samples:
                break

In [15]:
def augment_image(data_path, label, aug_num):

    start_time = time.time()
    # path to save augmented image
    augmented_data_path_from = os.path.join(data_path, 'original', 'train')
    augmented_data_path_to = os.path.join(data_path, 'augmented', 'train')

    for i in range(0, len(label)):
        folder_from = os.path.join(augmented_data_path_from, label[i])
        folder_to = os.path.join(augmented_data_path_to, label[i])
        augment_data(file_dir=folder_from, n_generated_samples=aug_num[i], save_to_dir=folder_to)
    
    end_time = time.time()

    execution_time = (end_time - start_time)
    print(f"Elapsed time: {hms_string(execution_time)}")

    data_summary(augmented_data_path_to, label)
    

    augmented_data_path_from = os.path.join(data_path, 'original', 'val')
    augmented_data_path_to = os.path.join(data_path, 'augmented', 'val')

    for i in label:
        folderPath_from = os.path.join(augmented_data_path_from,i)
        folderPath_to = os.path.join(augmented_data_path_to,i)
        num = 1
        for file in tqdm(os.listdir(folderPath_from)):
            file = cv2.imread(os.path.join(folderPath_from, file))
            file = cv2.cvtColor(file, cv2.COLOR_BGR2RGB)      
            a = str(f'{i}-{num}.png')
            plt.imsave(os.path.join(folderPath_to, a), file)
            num = num + 1

In [13]:
def data_summary(main_path, label):
    

    m = 0
    num = []
    for i in label:
        path = os.path.join(main_path, i)
        num.append(len(listdir(path)))
        m = m + len(listdir(path))

    print(f"Number of examples: {m}")
    for i in range(0, len(label)):
        prec = (num[i]* 100.0) / m
        print(f"Percentage of {label[i]} examples: {prec}%, number of {label[i]} examples: {num[i]}")

In [16]:
augment_image('/root/autodl-tmp/datasets', ['cat', 'dog'], [3, 3])

100%|██████████| 1000/1000 [01:08<00:00, 14.53it/s]
